In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


sys.path.append(os.path.abspath("../src"))

from cleaning import audit_and_clean_data
from features import create_rfm_features
from evaluation import evaluate_clustering

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
%matplotlib inline

print("Libraries imported successfully.")

Matplotlib is building the font cache; this may take a moment.


Libraries imported successfully.


In [2]:
raw_data_path = "../data/raw/Online Retail.xlsx"


df_clean = audit_and_clean_data(raw_data_path)

print(f"Cleaned dataset shape: {df_clean.shape}")
print(f"Distinct Customers: {df_clean['CustomerID'].nunique()}")
print(f"Date Range: {df_clean['InvoiceDate'].min()} to {df_clean['InvoiceDate'].max()}")

Cleaned dataset shape: (397884, 9)
Distinct Customers: 4338
Date Range: 2010-12-01 08:26:00 to 2011-12-09 12:50:00


In [3]:

rfm_df = create_rfm_features(df_clean)
rfm_df.to_csv("../data/processed/customer_rfm.csv", index=False)

print("RFM Table Sample:")
display(rfm_df.head())

RFM Table Sample:


,CustomerID,Recency,Frequency,Monetary,Recency_log,Frequency_log,Monetary_log
0,12346,326,1,77183.60,5.789960,0.693147,11.253955
1,12347,2,7,4310.00,1.098612,2.079442,8.368925
2,12348,75,4,1797.24,4.330733,1.609438,7.494564
3,12349,19,1,1757.55,2.995732,0.693147,7.472245
4,12350,310,1,334.40,5.739793,0.693147,5.815324


In [4]:
features_to_scale = ['Recency_log', 'Frequency_log', 'Monetary_log']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(rfm_df[features_to_scale])

print(f"Scaled matrix shape: {X_scaled.shape}")

Scaled matrix shape: (4338, 3)


In [5]:
results = []
k_target = 4  

# 1. K-Means
kmeans = KMeans(n_clusters=k_target, random_state=RANDOM_STATE, n_init=10)
kmeans_labels = kmeans.fit_predict(X_scaled)
results.append(evaluate_clustering(X_scaled, kmeans_labels, "K-Means (k=4)"))

# 2. Agglomerative Hierarchical Clustering
agg = AgglomerativeClustering(n_clusters=k_target)
agg_labels = agg.fit_predict(X_scaled)
results.append(evaluate_clustering(X_scaled, agg_labels, "Agglomerative (k=4)"))

# 3. Gaussian Mixture Model (GMM)
gmm = GaussianMixture(n_components=k_target, random_state=RANDOM_STATE)
gmm_labels = gmm.fit_predict(X_scaled)
results.append(evaluate_clustering(X_scaled, gmm_labels, "GMM (k=4)"))

# Comparison Table 
comparison_df = pd.DataFrame(results)
comparison_df.to_csv("../outputs/tables/model_comparison.csv", index=False)
display(comparison_df)

,Model,Silhouette,Davies-Bouldin,Calinski-Harabasz,Clusters
0,K-Means (k=4),0.337134,1.010147,3328.993968,4
1,Agglomerative (k=4),0.241918,1.119958,2615.051629,4
2,GMM (k=4),0.173930,1.719092,2154.678155,4


In [6]:
rfm_df['Cluster'] = kmeans_labels


cluster_profile = rfm_df.groupby('Cluster').agg({
    'Recency': ['median', lambda x: x.quantile(0.75) - x.quantile(0.25)],
    'Frequency': ['median', lambda x: x.quantile(0.75) - x.quantile(0.25)],
    'Monetary': ['median', lambda x: x.quantile(0.75) - x.quantile(0.25)],
    'CustomerID': 'count'
}).reset_index()

cluster_profile.columns = [
    'Cluster', 'Recency_Median', 'Recency_IQR', 
    'Freq_Median', 'Freq_IQR', 'Monetary_Median', 'Monetary_IQR', 'Customer_Count'
]
cluster_profile['Customer_Share_%'] = (cluster_profile['Customer_Count'] / len(rfm_df)) * 100

cluster_profile.to_csv("../outputs/tables/cluster_profiles.csv", index=False)
display(cluster_profile)

,Cluster,Recency_Median,Recency_IQR,Freq_Median,Freq_IQR,Monetary_Median,Monetary_IQR,Customer_Count,Customer_Share_%
0,0,17.0,18.0,2.0,2.0,471.700,433.4200,837,19.294606
1,1,8.0,13.0,10.0,8.0,3733.870,4017.1125,716,16.505302
2,2,56.0,58.0,4.0,2.0,1345.620,1102.0600,1173,27.040111
3,3,177.0,183.0,1.0,1.0,298.255,274.3125,1612,37.159982
